In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from scipy.sparse import csr_matrix, save_npz
import joblib
import os


In [8]:
# ═══════════════════════════════════════════════════════════════
# STEP 1 — Load & Clean Interactions
# ═══════════════════════════════════════════════════════════════
df = pd.read_csv("myket.csv", index_col=False)
df.columns = ["user_id", "app_id", "timestamp", "state_label", "features"]
df = df[["user_id", "app_id", "timestamp"]].copy()
df["timestamp"] = pd.to_numeric(df["timestamp"])
df = df.sort_values(["user_id", "timestamp"]).reset_index(drop=True)

print(df.head(10))
print(f"Loaded {len(df):,} interactions")
print(f"Users: {df['user_id'].nunique():,} | Apps: {df['app_id'].nunique():,}")


/tmp/ipykernel_14970/1652807071.py:4: ParserWarning: Length of header or names does not match length of data. This leads to a loss of data with index_col=False.
  df = pd.read_csv("myket.csv", index_col=False)


   user_id  app_id    timestamp
0        0     799  8716679.890
1        0    6242  8717003.433
2        0    2728  8717393.770
3        0    7158  8717530.477
4        0    4214  8718073.093
5        0    3688  8718085.007
6        0    5418  8767554.923
7        0    5948  8771614.620
8        0    7104  8771806.477
9        0    5141  8772753.730
Loaded 694,121 interactions
Users: 10,000 | Apps: 7,988


In [9]:
# ═══════════════════════════════════════════════════════════════
# STEP 2 — Load & Normalize App Features (zero vector fallback)
# ═══════════════════════════════════════════════════════════════
app_features = np.load("app_info_sample.npy", allow_pickle=True)  # (7988, 33)

num_features = app_features[:, :3].copy()   # install count, avg rating, rating count
cat_features = app_features[:, 3:].copy()   # 30 one-hot category dummies

# Log-scale skewed numerical features
num_features[:, 0] = np.log1p(num_features[:, 0])  # log(installs)
num_features[:, 2] = np.log1p(num_features[:, 2])  # log(rating count)

# Identify apps WITH features (not all-zero rows)
has_features = ~np.all(app_features == 0, axis=1)
print(f"\nApps with features:  {has_features.sum():,} / {len(app_features):,}")
print(f"Apps using zero fallback: {(~has_features).sum():,}")

# Fit scaler ONLY on apps that have features
scaler = StandardScaler()
num_features[has_features] = scaler.fit_transform(num_features[has_features])
# Apps without features remain as zero vectors — neutral fallback

# Final feature matrix
app_feature_matrix = np.hstack([num_features, cat_features]).astype(np.float32)
print(f"App feature matrix shape: {app_feature_matrix.shape}")  # (7988, 33)



Apps with features:  7,988 / 7,988
Apps using zero fallback: 0
App feature matrix shape: (7988, 33)


In [10]:
# ═══════════════════════════════════════════════════════════════
# STEP 3 — User-Based Split (last N interactions per user)
#           Train: all but last N
#           Val:   second-to-last N interactions
#           Test:  last N interactions
# ═══════════════════════════════════════════════════════════════
N = 5  # held-out interactions per user for test, same for val

train_rows, val_rows, test_rows = [], [], []

for user_id, group in df.groupby("user_id"):
    group = group.sort_values("timestamp")
    n = len(group)

    if n < N * 2 + 1:
        # Not enough interactions — put everything in train
        train_rows.append(group)
        continue

    train_rows.append(group.iloc[:-(N * 2)])
    val_rows.append(group.iloc[-(N * 2):-N])
    test_rows.append(group.iloc[-N:])

train_df = pd.concat(train_rows).reset_index(drop=True)
val_df   = pd.concat(val_rows).reset_index(drop=True)
test_df  = pd.concat(test_rows).reset_index(drop=True)

print(f"\nN = {N} held-out interactions per user per split")
print(f"  Train: {len(train_df):,} interactions | "
      f"{train_df['user_id'].nunique():,} users")
print(f"  Val:   {len(val_df):,} interactions  | "
      f"{val_df['user_id'].nunique():,} users")
print(f"  Test:  {len(test_df):,} interactions  | "
      f"{test_df['user_id'].nunique():,} users")



N = 5 held-out interactions per user per split
  Train: 594,121 interactions | 10,000 users
  Val:   50,000 interactions  | 10,000 users
  Test:  50,000 interactions  | 10,000 users


In [11]:
# ═══════════════════════════════════════════════════════════════
# STEP 4 — User History Dict (train interactions per user)
#           Used by both models as context / seen-items mask
# ═══════════════════════════════════════════════════════════════
user_train_history = (
    train_df.sort_values("timestamp")
    .groupby("user_id")["app_id"]
    .apply(list)
    .to_dict()
)

# Sanity check
sample_user = list(user_train_history.keys())[0]
print(f"\nSample user {sample_user} train history "
      f"({len(user_train_history[sample_user])} interactions):")
print(f"  {user_train_history[sample_user][:10]} ...")



Sample user 0 train history (35 interactions):
  [799, 6242, 2728, 7158, 4214, 3688, 5418, 5948, 7104, 5141] ...


In [12]:
# ═══════════════════════════════════════════════════════════════
# STEP 5 — Interaction Matrix (for Hybrid / CF model)
#           Built from TRAIN only — no leakage
# ═══════════════════════════════════════════════════════════════
n_users = df["user_id"].max() + 1   # 10,000
n_items = df["app_id"].max() + 1    # 7,988

train_matrix = csr_matrix(
    (np.ones(len(train_df)),
     (train_df["user_id"].values, train_df["app_id"].values)),
    shape=(n_users, n_items)
)

print(f"\nInteraction matrix: {train_matrix.shape}")
print(f"Sparsity: {1 - train_matrix.nnz / (n_users * n_items):.4%}")




Interaction matrix: (10000, 7988)
Sparsity: 99.4064%


In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 6 — Save All Pipeline Outputs
# ═══════════════════════════════════════════════════════════════
os.makedirs("pipeline_output", exist_ok=True)

train_df.to_csv("pipeline_output/train.csv", index=False)
val_df.to_csv("pipeline_output/val.csv",     index=False)
test_df.to_csv("pipeline_output/test.csv",   index=False)

np.save("pipeline_output/app_feature_matrix.npy", app_feature_matrix)
save_npz("pipeline_output/train_interaction_matrix.npz", train_matrix)
joblib.dump(scaler, "pipeline_output/feature_scaler.pkl")
joblib.dump(user_train_history, "pipeline_output/user_train_history.pkl")

print("\n── Pipeline outputs saved ──")
print("  train.csv                     → temporal model + CF")
print("  val.csv                       → validation")
print("  test.csv                      → final evaluation")
print("  app_feature_matrix.npy        → hybrid model content features")
print("  train_interaction_matrix.npz  → hybrid model CF component")
print("  feature_scaler.pkl            → consistent feature scaling")
print("  user_train_history.pkl        → context window / seen-items mask")